# Art Image Downloader — Wikimedia Commons

Downloads painting images from Wikimedia Commons for NST style training data.

## How to use for a different artist set

1. Edit the **Configuration** cell (cell 2) — change `OUTPUT_SUBDIR` and `ARTWORKS`.
2. Run all cells.

### Example: Mondrian
```python
OUTPUT_SUBDIR = "mondrian"
ARTWORKS = [
    {"artist": "Mondrian", "title": "Composition II in Red Blue and Yellow", "year": 1930},
    {"artist": "Mondrian", "title": "Tableau I",                              "year": 1921},
    {"artist": "Mondrian", "title": "Broadway Boogie Woogie",                 "year": 1943},
    {"artist": "Mondrian", "title": "Composition with Red Yellow and Blue",   "year": 1921},
]
```

> **Tip:** Wikimedia Commons file names are unpredictable — the notebook tries
> several query variants and ranks hits by artist/title overlap before downloading.

In [11]:
# === CONFIGURATION — edit this cell to download different artwork sets ===

OUTPUT_SUBDIR = "synthetic-cubism"   # subfolder under sample_images/

# Synthetic Cubism (~1913–1921): flat bold shapes, bright colours, collage feel.
# Original 4 from painter_style_creation.md + 4 additional representative works:
#   Picasso "Guitar" (1913)              — first major Synthetic Cubist sculpture/painting hybrid
#   Gris    "Violin and Engraving" (1913) — strong interlocking planes, muted palette
#   Léger   "Three Women" (1921)         — bold tubular figures, high colour contrast
#   Gris    "The Sunblind" (1914)        — rich texture, typical collage-like surface

ARTWORKS = [
    # --- from painter_style_creation.md ---
    {"artist": "Picasso", "title": "Harlequin",           "year": 1915},
    {"artist": "Picasso", "title": "Three Musicians",      "year": 1921},
    {"artist": "Léger",   "title": "The City",             "year": 1919},
    {"artist": "Gris",    "title": "Guitar and Flowers",   "year": 1912},
    # --- additional representative Synthetic Cubist works ---
    {"artist": "Picasso", "title": "Guitar",               "year": 1913},
    {"artist": "Gris",    "title": "Violin and Engraving", "year": 1913},
    {"artist": "Léger",   "title": "Three Women",          "year": 1921},
    {"artist": "Gris",    "title": "The Sunblind",         "year": 1914},
]


In [12]:
# Imports and path setup
import re
import sys
import time
import warnings
from collections import Counter
from pathlib import Path
from typing import Optional

try:
    import requests
    import urllib3
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests, urllib3

# Corporate networks often use SSL-inspection proxies — disable cert verification
# and suppress the resulting InsecureRequestWarning.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
VERIFY_SSL = False   # set True if you trust the system CA bundle

# Locate repo root — works whether kernel cwd is scripts/ or workspace root
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "scripts" else _cwd

OUTPUT_DIR = REPO_ROOT / "sample_images" / OUTPUT_SUBDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMONS_API   = "https://commons.wikimedia.org/w/api.php"
HEADERS       = {"User-Agent": "StyleTransferArtDownloader/1.0 (educational, offline NST training)"}
THUMB_WIDTH   = 1600   # px — good resolution for NST training
REQUEST_DELAY = 0.6    # seconds between API calls (polite crawling)

print(f"Repo root        : {REPO_ROOT}")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Artworks to fetch: {len(ARTWORKS)}")
print(f"SSL verify       : {VERIFY_SSL}")


Repo root        : c:\Users\i09300076\OneDrive - Endress+Hauser\DEV\Python3\style_transfer
Output directory : c:\Users\i09300076\OneDrive - Endress+Hauser\DEV\Python3\style_transfer\sample_images\synthetic-cubism
Artworks to fetch: 8
SSL verify       : False


In [16]:
# Search & download helpers

def _commons_search(query: str, limit: int = 8) -> list:
    """Full-text search in Wikimedia Commons File namespace (ns=6)."""
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srnamespace": "6",
        "format": "json",
        "srlimit": str(limit),
    }
    for attempt in range(4):
        r = requests.get(COMMONS_API, params=params, headers=HEADERS, timeout=15, verify=VERIFY_SSL)
        if r.status_code == 429:
            wait = 8 * (attempt + 1)
            print(f"\n    [API rate limit, waiting {wait}s ...]", end="", flush=True)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json().get("query", {}).get("search", [])
    return []


def _get_image_info(file_title: str) -> Optional[dict]:
    """Return imageinfo dict for a Commons file title, or None."""
    params = {
        "action": "query",
        "titles": file_title,
        "prop": "imageinfo",
        "iiprop": "url|size|mime",
        "iiurlwidth": str(THUMB_WIDTH),
        "format": "json",
    }
    for attempt in range(4):
        r = requests.get(COMMONS_API, params=params, headers=HEADERS, timeout=15, verify=VERIFY_SSL)
        if r.status_code == 429:
            wait = 8 * (attempt + 1)
            print(f"\n    [API rate limit, waiting {wait}s ...]", end="", flush=True)
            time.sleep(wait)
            continue
        r.raise_for_status()
        for page in r.json().get("query", {}).get("pages", {}).values():
            ii_list = page.get("imageinfo", [])
            if ii_list:
                return ii_list[0]
        return None
    return None


def _score_hit(file_title: str, artist: str, title: str) -> int:
    """Score a Commons search result by relevance (higher = better)."""
    lower = file_title.lower()
    score = 0
    if artist.lower() in lower:
        score += 3
    for word in re.split(r"\W+", title.lower()):
        if len(word) >= 4 and word in lower:
            score += 1
    return score


def _candidate_queries(artist: str, title: str, year) -> list:
    """Ordered list of search queries, from most to least specific."""
    title_plain = re.sub(r"[',]", "", title)
    queries = [
        f"{artist} {title} {year}",
        f"{artist} {title_plain}",
        f"{artist} {title}",
        f"{title_plain} {artist}",
        f"{title_plain}",
    ]
    seen: set = set()
    return [q for q in queries if not (q in seen or seen.add(q))]


def search_artwork(artist: str, title: str, year=None, verbose: bool = False):
    """Search Wikimedia Commons for a painting.

    Returns (download_url, commons_file_title) or None if nothing usable found.
    """
    best_score = -1
    best_url   = None
    best_title = None

    for query in _candidate_queries(artist, title, year):
        hits = _commons_search(query)
        time.sleep(REQUEST_DELAY)

        for hit in hits:
            file_title = hit["title"]
            if not file_title.startswith("File:"):
                continue
            ext = Path(file_title).suffix.lower()
            if ext not in (".jpg", ".jpeg", ".png"):
                continue

            score = _score_hit(file_title, artist, title)
            if score <= best_score:
                continue

            ii = _get_image_info(file_title)
            time.sleep(REQUEST_DELAY)
            if not ii:
                continue
            if not ii.get("mime", "").startswith("image/"):
                continue
            url = ii.get("thumburl") or ii.get("url")
            if not url:
                continue
            if verbose:
                print(f"      candidate (score={score}): {file_title}")
            best_score = score
            best_url   = url
            best_title = file_title

        if best_score >= 3:
            break

    return (best_url, best_title) if best_url else None


def download_image(url: str, dest: Path) -> bool:
    """Stream-download image from url to dest, with CDN 429 retry."""
    for attempt in range(5):
        try:
            r = requests.get(url, headers=HEADERS, timeout=60, stream=True, verify=VERIFY_SSL)
            if r.status_code == 429:
                wait = 15 * (attempt + 1)
                print(f"\n    [CDN rate limit, waiting {wait}s ...]", end="", flush=True)
                time.sleep(wait)
                continue
            r.raise_for_status()
            dest.write_bytes(r.content)
            return True
        except requests.exceptions.HTTPError as exc:
            if exc.response is not None and exc.response.status_code == 429:
                wait = 15 * (attempt + 1)
                print(f"\n    [CDN rate limit, waiting {wait}s ...]", end="", flush=True)
                time.sleep(wait)
                continue
            print(f"    download error: {exc}")
            return False
        except Exception as exc:
            print(f"    download error: {exc}")
            return False
    print(f"    download failed after retries")
    return False


def safe_filename(artist: str, title: str, year, ext: str = ".jpg") -> str:
    name = f"{artist}_{title}" + (f"_{year}" if year else "")
    name = re.sub(r"[^\w\s,'-]", "", name)
    name = re.sub(r"[\s,]+", "_", name.strip())
    return name + ext


print("Helper functions ready.")


Helper functions ready.


In [17]:
# Run downloads

results: list = []

for artwork in ARTWORKS:
    artist = artwork["artist"]
    title  = artwork["title"]
    year   = artwork.get("year")

    print(f"  Searching : {artist} — {title} ({year or '?'}) ...", end="  ", flush=True)

    found = search_artwork(artist, title, year)

    if found is None:
        print("NOT FOUND")
        results.append(dict(artist=artist, title=title, year=year,
                            status="not_found", file=None, source=None))
        continue

    url, commons_title = found
    ext  = Path(commons_title).suffix.lower() or ".jpg"
    dest = OUTPUT_DIR / safe_filename(artist, title, year, ext)

    if dest.exists():
        kb = dest.stat().st_size // 1024
        print(f"already exists  ({kb} KB)  {dest.name}")
        results.append(dict(artist=artist, title=title, year=year,
                            status="already_exists", file=str(dest), source=commons_title))
        continue

    ok = download_image(url, dest)
    if ok:
        kb = dest.stat().st_size // 1024
        print(f"downloaded  ({kb} KB)  {dest.name}")
        results.append(dict(artist=artist, title=title, year=year,
                            status="downloaded", file=str(dest), source=commons_title))
    else:
        results.append(dict(artist=artist, title=title, year=year,
                            status="download_failed", file=None, source=commons_title))

  Searching : Picasso — Harlequin (1915) ...  already exists  (574 KB)  Picasso_Harlequin_1915.jpg
  Searching : Picasso — Three Musicians (1921) ...  already exists  (41 KB)  Picasso_Three_Musicians_1921.jpeg
  Searching : Léger — The City (1919) ...  downloaded  (77 KB)  Léger_The_City_1919.jpg
  Searching : Gris — Guitar and Flowers (1912) ...  
    [API rate limit, waiting 8s ...]
    [API rate limit, waiting 16s ...]
    [API rate limit, waiting 24s ...]already exists  (113 KB)  Gris_Guitar_and_Flowers_1912.jpg
  Searching : Picasso — Guitar (1913) ...  already exists  (1579 KB)  Picasso_Guitar_1913.jpg
  Searching : Gris — Violin and Engraving (1913) ...  
    [API rate limit, waiting 8s ...]
    [API rate limit, waiting 16s ...]
    [API rate limit, waiting 24s ...]already exists  (149 KB)  Gris_Violin_and_Engraving_1913.jpg
  Searching : Léger — Three Women (1921) ...  already exists  (632 KB)  Léger_Three_Women_1921.png
  Searching : Gris — The Sunblind (1914) ...  downloaded 

In [18]:
# Statistics summary

counts = Counter(r["status"] for r in results)
total  = len(results)
ICON   = {"downloaded": "OK", "already_exists": "==", "not_found": "XX", "download_failed": "!!"}

print(f"\n{'='*64}")
print(f"  Download summary  —  '{OUTPUT_SUBDIR}'")
print(f"{'='*64}")
print(f"  Total requested  : {total}")
print(f"  OK Downloaded    : {counts.get('downloaded', 0)}")
print(f"  == Already exists: {counts.get('already_exists', 0)}")
print(f"  XX Not found     : {counts.get('not_found', 0)}")
print(f"  !! Download fail : {counts.get('download_failed', 0)}")
print(f"{'─'*64}")
print(f"  {'St':<5}{'Artist':<12}{'Title':<44}{'Year'}")
print(f"{'─'*64}")
for r in results:
    icon     = ICON.get(r["status"], "??")
    year_str = str(r["year"]) if r["year"] else "—"
    title_s  = r["title"][:42] + "..." if len(r["title"]) > 42 else r["title"]
    print(f"  {icon:<5}{r['artist']:<12}{title_s:<44}{year_str}")
print(f"{'='*64}")

# Files on disk
files = sorted(OUTPUT_DIR.glob("*.*"))
print(f"\nFiles in {OUTPUT_DIR}  ({len(files)} total):")
for f in files:
    print(f"  {f.name:<58}  {f.stat().st_size // 1024:>5} KB")


  Download summary  —  'synthetic-cubism'
  Total requested  : 8
  OK Downloaded    : 2
  == Already exists: 6
  XX Not found     : 0
  !! Download fail : 0
────────────────────────────────────────────────────────────────
  St   Artist      Title                                       Year
────────────────────────────────────────────────────────────────
  ==   Picasso     Harlequin                                   1915
  ==   Picasso     Three Musicians                             1921
  OK   Léger       The City                                    1919
  ==   Gris        Guitar and Flowers                          1912
  ==   Picasso     Guitar                                      1913
  ==   Gris        Violin and Engraving                        1913
  ==   Léger       Three Women                                 1921
  OK   Gris        The Sunblind                                1914

Files in c:\Users\i09300076\OneDrive - Endress+Hauser\DEV\Python3\style_transfer\sample_images\synt